In [0]:
# =============================================================================
# segmentation_snapshots_report  -  READ ONLY. Browse test_segmentation_snapshots.
#  - INDEX table of every snapshot (total / ACTIVE / ARCHIVE / valid / QUARANTINED / clones / Δ vs prev)
#  - a DROPDOWN to SELECT any snapshot -> per-state detail split by TIER (active/archive)
#  - IN-PAGE DOWNLOAD buttons (data-URI): the HTML report AND a consolidated Excel
#    (snapshots_index + per_state_detail sheets) - one click from the notebook, no file hunting.
# Renders inline (displayHTML) + also saves .html + .xlsx to your Results folder. Writes no data.
# HOW: Run All once -> pick a run in the "snapshot" dropdown -> it re-renders.
# =============================================================================

In [0]:
# ---- CELL 0 : config + auth ----
SNAPSHOT_TABLE = "test_segmentation_snapshots"

from pyspark.sql import functions as F
from pyspark.sql.functions import *
import datetime, html as _h, io, base64
def esc(x): return _h.escape("" if x is None else str(x))
try:
    _c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
    env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
    KV=f"ingest{lz_key}-meta002-{env_name}"
    cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
    for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
        spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
        spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
        spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
        spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")
except Exception as e:
    print("auth note:", str(e)[:80])

In [0]:
# ---- CELL 1 : aggregate (with tier) + build the selector widget ----
if not spark.catalog.tableExists(SNAPSHOT_TABLE):
    displayHTML(f"<h3 style='font-family:sans-serif'>No snapshots yet</h3><p style='font-family:sans-serif'>"
                f"Table <code>{SNAPSHOT_TABLE}</code> does not exist. Run <b>segmentation_snapshot</b> first.</p>")
    dbutils.notebook.exit("no table")

snap = spark.table(SNAPSHOT_TABLE)
if "tier" not in snap.columns: snap = snap.withColumn("tier", lit("active"))

idx = (snap.groupBy("snapshot_id","data_cut_label","snapshot_datetime","note")
          .agg(count("*").alias("total"),
               sum(when(col("tier")=="active",1).otherwise(0)).alias("active"),
               sum(when(col("tier")=="archive",1).otherwise(0)).alias("archive"),
               sum(when(col("is_valid")==True,1).otherwise(0)).alias("valid"),
               sum(when(col("is_valid")==False,1).otherwise(0)).alias("quarantined"),
               sum(col("is_clone").cast("int")).alias("clones"),
               countDistinct("assigned_state").alias("states"))
          .orderBy(col("snapshot_datetime").desc()))
idx_pd = idx.toPandas().reset_index(drop=True)

det = (snap.groupBy("snapshot_id","tier","assigned_state")
           .agg(count("*").alias("cases"),
                sum(when(col("is_valid")==True,1).otherwise(0)).alias("valid"),
                sum(when(col("is_valid")==False,1).otherwise(0)).alias("quarantined"),
                sum(col("is_clone").cast("int")).alias("clones")))
det_pd = det.toPandas()

labels = list(idx_pd["data_cut_label"])
try:
    dbutils.widgets.dropdown("snapshot", labels[0], labels[:1024], "Snapshot to view")
except Exception as e:
    print("widget note:", str(e)[:80])

In [0]:
# ---- CELL 2 : render HTML (index + tier detail) + in-page DOWNLOAD buttons ----
try:    sel = dbutils.widgets.get("snapshot")
except Exception: sel = labels[0]
selrow = idx_pd[idx_pd["data_cut_label"]==sel]
if selrow.empty: selrow = idx_pd.head(1); sel = selrow.iloc[0]["data_cut_label"]
sel_id = selrow.sort_values("snapshot_datetime").iloc[-1]["snapshot_id"]

_bytime = idx_pd.sort_values("snapshot_datetime"); _last=None; delta_map={}
for _, r in _bytime.iterrows():
    delta_map[r["snapshot_id"]] = "" if _last is None else f"{int(r['total'])-_last:+d}"; _last=int(r["total"])

CSS = """<style>
.segrep{font-family:-apple-system,Segoe UI,Roboto,sans-serif;color:#1a1a1a;max-width:1150px}
.segrep h1{font-size:20px;margin:0 0 2px} .segrep h2{font-size:15px;margin:20px 0 6px;color:#333}
.segrep .sub{color:#666;font-size:12px;margin:0 0 12px}
.segrep table{border-collapse:collapse;font-size:12.5px;margin:4px 0 10px;width:100%}
.segrep th,.segrep td{border:1px solid #e2e2e2;padding:5px 8px;text-align:right}
.segrep th{background:#f4f6f8;font-weight:600}
.segrep td.l,.segrep th.l{text-align:left}
.segrep tr:nth-child(even) td{background:#fafbfc}
.segrep tr.sel td{background:#fff6df !important}
.segrep tr.hasfail td{background:#fdecec}
.segrep tr.arch td{background:#eef6ff}
.segrep .pos{color:#0a7a28} .segrep .neg{color:#b3261e}
.segrep .q{color:#b3261e;font-weight:600} .segrep .q0{color:#9a9a9a}
.segrep .clone{color:#8a5a00} .segrep .chip{background:#eef3ff;border:1px solid #d7e2ff;border-radius:10px;padding:1px 7px;font-size:11px}
.segrep a.dl{display:inline-block;background:#0b5cad;color:#fff;text-decoration:none;padding:7px 14px;border-radius:6px;font-size:13px;margin:0 8px 10px 0}
.segrep a.dl:hover{background:#094a8c}
</style>"""
def td(v,cls=""): return f'<td class="{cls}">{esc(v)}</td>'

# INDEX
rows=[]
for _, r in idx_pd.iterrows():
    d=delta_map.get(r["snapshot_id"],""); dcls="pos" if d.startswith("+") else ("neg" if d.startswith("-") else "")
    q=int(r["quarantined"]); sel_cls="sel" if r["snapshot_id"]==sel_id else ""
    rows.append(f'<tr class="{sel_cls}">'
      f'<td class="l"><b>{esc(r["data_cut_label"])}</b></td><td class="l">{esc(r["snapshot_datetime"])}</td>'
      f'<td class="l">{esc(r["note"])}</td>{td(int(r["total"]))}<td class="{dcls}">{esc(d)}</td>'
      f'{td(int(r["active"]))}{td(int(r["archive"]))}'
      f'{td(int(r["valid"]))}<td class="{"q" if q else "q0"}">{q}</td>'
      f'<td class="clone">{esc(int(r["clones"]))}</td>{td(int(r["states"]))}'
      f'<td class="l" style="color:#999">{esc(r["snapshot_id"])[:8]}</td></tr>')
index_html=('<table><tr><th class="l">label</th><th class="l">datetime</th><th class="l">note</th>'
 '<th>total</th><th>Δ prev</th><th>active</th><th>archive</th><th>valid</th><th>quarantined</th><th>clones</th><th>states</th><th class="l">id</th></tr>'
 +"".join(rows)+"</table>")

# SELECTED snapshot per-state detail, split by tier
sub=det_pd[det_pd["snapshot_id"]==sel_id].sort_values(["tier","assigned_state"])
drows=[]
for tier_name in ["active","archive"]:
    tsub=sub[sub["tier"]==tier_name]
    if tsub.empty: continue
    drows.append(f'<tr><td class="l" colspan="5" style="background:#eef2f6"><b>{tier_name.upper()}</b></td></tr>')
    for _, rr in tsub.iterrows():
        q=int(rr["quarantined"]); cls="hasfail" if q>0 else ("arch" if tier_name=="archive" else "")
        drows.append(f'<tr class="{cls}"><td class="l">{esc(rr["assigned_state"])}</td>{td(int(rr["cases"]))}'
          f'{td(int(rr["valid"]))}<td class="{"q" if q else "q0"}">{q}</td><td class="clone">{esc(int(rr["clones"]))}</td></tr>')
    tt=tsub[["cases","valid","quarantined","clones"]].sum()
    drows.append(f'<tr><td class="l"><b>{tier_name} TOTAL</b></td>{td(int(tt["cases"]))}{td(int(tt["valid"]))}'
                 f'<td class="q">{int(tt["quarantined"])}</td><td class="clone">{int(tt["clones"])}</td></tr>')
detail_html=(f'<h2>Selected: {esc(sel)} <span class="chip">{esc(selrow.iloc[0]["snapshot_datetime"])}</span></h2>'
 '<table><tr><th class="l">state / segment</th><th>cases</th><th>valid</th><th>quarantined</th><th>clones</th></tr>'
 +"".join(drows)+"</table>")

report_body=(f'<div class="segrep">{CSS}<h1>Segmentation snapshots — recorded baselines</h1>'
 f'<p class="sub">{len(idx_pd)} snapshot(s) in <code>{esc(SNAPSHOT_TABLE)}</code>. '
 f'Active + archive tiers. Use the <b>snapshot</b> dropdown to pick a run; per-state detail (with quarantined) shows below. '
 f'Red = states with failures; blue = archive.</p><h2>All snapshots</h2>{index_html}{detail_html}</div>')

# ---- build downloadable artefacts (data-URI) ----
buttons=""
try:
    try: import openpyxl  # noqa
    except Exception:
        import subprocess,sys; subprocess.run([sys.executable,"-m","pip","install","-q","openpyxl"])
    buf=io.BytesIO()
    with __import__("pandas").ExcelWriter(buf, engine="openpyxl") as xw:
        idx_pd.to_excel(xw, sheet_name="snapshots_index", index=False)
        det_pd.to_excel(xw, sheet_name="per_state_detail", index=False)
    xls_b64=base64.b64encode(buf.getvalue()).decode()
    html_doc="<!doctype html><meta charset='utf-8'>"+report_body
    html_b64=base64.b64encode(html_doc.encode("utf-8")).decode()
    stamp=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    buttons=(f'<div style="margin:8px 0 4px">'
      f'<a class="dl" download="segmentation_snapshots_{stamp}.html" '
      f'href="data:text/html;base64,{html_b64}">⬇ Download HTML report</a>'
      f'<a class="dl" download="segmentation_snapshots_{stamp}.xlsx" '
      f'href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{xls_b64}">⬇ Download Excel (all snapshots)</a>'
      f'</div>')
except Exception as e:
    buttons=f'<div style="color:#b3261e;font-size:12px">download build note: {esc(str(e)[:120])}</div>'

displayHTML(f'<div class="segrep">{CSS}{buttons}</div>'+report_body)

# also save to Results (fallback / archival)
try:
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/segmentation_snapshots_report/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    open(f"{folder}/snapshots_report.html","w",encoding="utf-8").write("<!doctype html><meta charset='utf-8'>"+report_body)
    try:
        with __import__("pandas").ExcelWriter(f"{folder}/segmentation_snapshots.xlsx", engine="openpyxl") as xw:
            idx_pd.to_excel(xw, sheet_name="snapshots_index", index=False)
            det_pd.to_excel(xw, sheet_name="per_state_detail", index=False)
    except Exception: pass
    print("saved report ->", folder, "| selected:", sel)
except Exception as e:
    print("save note:", str(e)[:100])